
# Lab Session: LSTM for Easy Text Prediction

This notebook is designed for a beginner-friendly lab session on **LSTM for text data**.

## Learning goals
By the end of this lab, students will be able to:
- understand text as sequence data
- tokenize text into numbers
- create input-output sequences
- train an LSTM model for next-word prediction
- generate short text using the trained model

## Dataset used
A **small custom text corpus** is used so that the lab:
- runs quickly
- is easy to understand
- does not require downloading datasets
- works well in most college lab environments



## 1. Import required libraries
We will use TensorFlow/Keras for building the LSTM model and NumPy for prediction handling.


In [ ]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)



## 2. Create a small text dataset

This is a simple teacher-controlled corpus.  
Students can later add their own sentences and observe how the predictions change.


In [ ]:

corpus = [
    "deep learning is powerful",
    "deep learning is fun",
    "machine learning is interesting",
    "artificial intelligence is the future",
    "i love machine learning",
    "lstm is used for sequence prediction",
    "neural networks learn patterns",
    "data science is exciting",
    "python is easy to learn",
    "models improve with more data"
]

print("Number of sentences:", len(corpus))
print("\nSample corpus:")
for line in corpus:
    print("-", line)



## 3. Tokenization

A tokenizer converts words into integers.

Example:
- `deep` → 1
- `learning` → 2

The exact numbers depend on the fitted vocabulary.


In [ ]:

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

word_index = tokenizer.word_index
total_words = len(word_index) + 1  # +1 because indexing starts from 1

print("Vocabulary size:", total_words)
print("\nWord index:")
print(word_index)



## 4. Create training sequences

For each sentence, we create multiple smaller sequences.

Example for:
`deep learning is powerful`

We create:
- `deep learning`
- `deep learning is`
- `deep learning is powerful`

Then the model learns:
- input = all words except last
- output = last word


In [ ]:

input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total training sequences:", len(input_sequences))
print("\nSome sequences before padding:")
for seq in input_sequences[:10]:
    print(seq)



## 5. Pad the sequences

Not all sequences have the same length, so we pad them with zeros at the beginning.

This makes every input sequence have the same size.


In [ ]:

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

print("Maximum sequence length:", max_seq_len)
print("\nPadded sequences:")
print(input_sequences[:10])



## 6. Split into input (`X`) and output (`y`)

- `X` = all words except the last word
- `y` = the last word to be predicted

We convert `y` into one-hot encoding because this is a multiclass classification problem.


In [ ]:

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)



## 7. Build the LSTM model

### Model architecture
- **Embedding layer**: converts word indices into dense vectors
- **LSTM layer**: learns sequence patterns and context
- **Dense layer with softmax**: predicts the next word


In [ ]:

model = Sequential([
    Embedding(input_dim=total_words, output_dim=10, input_length=max_seq_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()



## 8. Train the model

Since the dataset is very small, we train for more epochs so the model can learn the patterns better.


In [ ]:

history = model.fit(X, y, epochs=200, verbose=1)



## 9. Function for next-word text generation

This function:
1. takes a seed text
2. converts it into tokens
3. pads it
4. predicts the next word
5. appends the predicted word to the sentence


In [ ]:

def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text



## 10. Test the model

Try a few seed texts and observe the output.


In [ ]:

print(generate_text("deep learning", 2))
print(generate_text("machine learning", 2))
print(generate_text("python is", 2))



## 11. Student activity

Students can now try their own seed texts.


In [ ]:

# Try your own inputs here
print(generate_text("artificial intelligence", 2))
print(generate_text("data science", 2))
print(generate_text("lstm is", 3))



## 12. Mini experiments for students

### Experiment 1
Change:
- `epochs = 200` to `epochs = 50`
- observe the output quality

### Experiment 2
Change:
- `LSTM(100)` to `LSTM(50)` or `LSTM(150)`
- compare results

### Experiment 3
Add new lines to the corpus such as:
- `"deep learning uses neural networks"`
- `"python is useful for machine learning"`

Then retrain and observe how predictions change.

### Experiment 4
Try different starting texts:
- `"deep"`
- `"models"`
- `"i love"`



## 13. Viva questions

1. What is sequence data?
2. Why is LSTM better than a simple RNN for long sequences?
3. What is tokenization?
4. Why do we pad sequences?
5. What does the embedding layer do?
6. What is the role of the softmax layer?
7. What does the model predict in this notebook?



## 14. Lab conclusion

In this lab, we learned how to:
- prepare text data for sequence modeling
- convert words into tokens
- build training samples for next-word prediction
- train an LSTM model
- generate short text sequences

This is one of the simplest and most useful introductory labs for text-based deep learning.
